# Audio Harness — подготовка датасета denoise

Этот ноутбук сохраняет всё **в Google Drive**, поэтому Colab можно оставить на ночь. Сейчас он скачивает MUSAN по переданной Google Drive-ссылке и достаёт только реальные шумы. Твой локальный `ru_RU.tgz` намеренно не скачивается повторно.

Не запускай одновременно с ним второй скачиватель MUSAN в браузере: они будут делить канал.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install gdown soundfile

In [ ]:
# Можно менять только DATA_ROOT, если хочешь другую папку на Google Drive.
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/audio-harness')
MUSAN_FILE_ID = '1kngNXIxSjXTV2JtS9PAbE03yQcnx7iu_'  # ссылка, которую тебе прислали
MAILABS_URL = 'https://www.caito.de/data/Training/stt_tts/ru_RU.tgz'
PREPARE_MAILABS = False  # включим позже, когда ты положишь локальный ru_RU.tgz в Google Drive
PAIR_COUNT = 500       # тестовых пар: хватает для первого сравнения
SEED = 42

RAW = DATA_ROOT / 'raw'
MUSAN_ARCHIVE = RAW / 'musan' / 'musan.zip'  # по присланной ссылке ZIP
MUSAN_NOISE = RAW / 'musan' / 'musan' / 'noise'
MAILABS_DIR = RAW / 'm_ailabs_ru'
MAILABS_ARCHIVE = RAW / 'm_ailabs_ru.tgz'
for path in (RAW / 'musan', MAILABS_DIR, DATA_ROOT / 'indexes', DATA_ROOT / 'splits', DATA_ROOT / 'derived'):
    path.mkdir(parents=True, exist_ok=True)
print('Результат будет лежать в:', DATA_ROOT)

## 1. MUSAN: скачать и распаковать только шумы

Если ячейка оборвётся, просто запусти её ещё раз: уже готовые части не удаляются. После успешной распаковки архив удаляется — на обычном Google Drive так остаётся намного больше места.

In [ ]:
import subprocess
from zipfile import ZipFile

if not MUSAN_NOISE.exists() or not any(MUSAN_NOISE.rglob('*.wav')):
    if not MUSAN_ARCHIVE.exists() or MUSAN_ARCHIVE.stat().st_size < 1_000_000_000:
        print('Скачиваю MUSAN из Google Drive…')
        subprocess.run(['gdown', '--fuzzy', f'https://drive.google.com/file/d/{MUSAN_FILE_ID}/view', '-O', str(MUSAN_ARCHIVE)], check=True)
    print('Распаковываю из ZIP только папку noise…')
    with ZipFile(MUSAN_ARCHIVE) as archive:
        members = [name for name in archive.namelist() if '/noise/' in name and not name.endswith('/')]
        assert members, 'В ZIP не найдена папка noise — пришли скрин содержимого архива.'
        for member in members:
            archive.extract(member, RAW / 'musan')
    MUSAN_ARCHIVE.unlink()  # оставляем только нужные noise WAV
else:
    print('Шумы MUSAN уже распакованы.')
print('WAV-шумов:', sum(1 for _ in MUSAN_NOISE.rglob('*.wav')))

## 2. Чистая русская речь: M-AILABS

У тебя этот архив уже есть локально, поэтому эта часть сейчас выключена. Когда понадобится обучение в Colab, просто загрузи `ru_RU.tgz` в `MyDrive/audio-harness/raw/`, затем поставь `PREPARE_MAILABS = True` в ячейке настроек.

In [ ]:
if not PREPARE_MAILABS:
    print('Пропуск: используем локальный M-AILABS, ничего не скачиваем повторно.')
elif not (MAILABS_DIR / 'ru_RU' / 'by_book').exists():
    if not MAILABS_ARCHIVE.exists() or MAILABS_ARCHIVE.stat().st_size < 1_000_000_000:
        print('Скачиваю M-AILABS ru_RU…')
        subprocess.run(['wget', '-c', '--show-progress', '-O', str(MAILABS_ARCHIVE), MAILABS_URL], check=True)
    print('Распаковываю M-AILABS…')
    subprocess.run(['tar', '-xzf', str(MAILABS_ARCHIVE), '-C', str(MAILABS_DIR)], check=True)
    MAILABS_ARCHIVE.unlink()  # распакованные WAV уже лежат на Drive
else:
    print('M-AILABS уже распакован.')

In [ ]:
# Инвентаризация и book-level split. Внутри одной книги клипы не пересекаются между split'ами.
import csv, random
from collections import defaultdict

wav_files = sorted(MAILABS_DIR.rglob('*.wav'))
if not wav_files:
    print('Пропуск split: M-AILABS пока не добавлен в Google Drive.')
else:
    index_path = DATA_ROOT / 'indexes' / 'm_ailabs_ru.csv'
    rows = []
    for wav in wav_files:
        rel = wav.relative_to(MAILABS_DIR)
        parts = rel.parts; marker = parts.index('by_book')
        gender, speaker, book = parts[marker + 1: marker + 4]
        rows.append({'dataset': 'm_ailabs_ru', 'path': str(rel), 'bytes': wav.stat().st_size, 'gender': gender, 'speaker': speaker, 'book': book})
    with index_path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)
    by_book = defaultdict(list)
    for row in rows: by_book[(row['gender'], row['speaker'], row['book'])].append(row)
    candidates = [key for key in by_book if sum(1 for k in by_book if k[:2] == key[:2]) > 1]
    rng = random.Random(SEED); rng.shuffle(candidates)
    assignment = {key: 'train' for key in by_book}; assignment[candidates[0]] = 'dev'; assignment[candidates[1]] = 'test'
    for split in ('train', 'dev', 'test'):
        output = DATA_ROOT / 'splits' / f'm_ailabs_ru_{split}.csv'
        split_rows = [{**r, 'split': split} for key, values in by_book.items() if assignment[key] == split for r in values]
        with output.open('w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=[*rows[0].keys(), 'split']); writer.writeheader(); writer.writerows(split_rows)
        print(split, len(split_rows), 'клипов')
    print('Всего:', len(rows), 'WAV; index:', index_path)

## 3. Честный eval-набор: реальные шумы + 0 / 5 / 10 dB SNR

Это не обучение модели, а фиксированный набор для сравнения baseline и будущего fine-tune. Для обучения шум лучше микшировать на лету, а не хранить десятки гигабайт одинаковых файлов.

In [ ]:
import numpy as np
import soundfile as sf

OUT = DATA_ROOT / 'derived' / 'denoise_eval'
manifest = OUT / 'manifest.csv'
if manifest.exists():
    print('Eval-набор уже есть:', manifest)
else:
    for name in ('clean', 'noise', 'noisy'): (OUT / name).mkdir(parents=True, exist_ok=True)
    with (DATA_ROOT / 'splits' / 'm_ailabs_ru_test.csv').open(encoding='utf-8') as f: clean_rows = list(csv.DictReader(f))
    noise_files = list(MUSAN_NOISE.rglob('*.wav'))
    assert noise_files, 'Не найдены WAV-шумы MUSAN'
    rng = random.Random(SEED); rng.shuffle(clean_rows)

    def one_channel(x): return x.mean(axis=1) if x.ndim == 2 else x
    def fit(x, length):
        if len(x) >= length:
            start = rng.randrange(len(x) - length + 1); return x[start:start + length]
        return np.tile(x, int(np.ceil(length / len(x))))[:length]
    def resample(x, old, new):
        if old == new: return x
        return np.interp(np.linspace(0, 1, round(len(x) * new / old), endpoint=False), np.linspace(0, 1, len(x), endpoint=False), x).astype(np.float32)
    def rms(x): return float(np.sqrt(np.mean(x.astype(np.float64) ** 2)))

    fields = ['id', 'clean', 'noise', 'noisy', 'snr_db', 'source_clean', 'source_noise']
    with manifest.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fields); writer.writeheader()
        for i, row in enumerate(clean_rows[:PAIR_COUNT]):
            clean, rate = sf.read(MAILABS_DIR / row['path'], dtype='float32', always_2d=False); clean = one_channel(clean)
            npath = rng.choice(noise_files); noise, nr = sf.read(npath, dtype='float32', always_2d=False)
            noise = fit(resample(one_channel(noise), nr, rate), len(clean))
            snr = rng.choice((0, 5, 10)); noise *= rms(clean) / max(rms(noise), 1e-8) / (10 ** (snr / 20))
            noisy = clean + noise; peak = max(float(np.max(np.abs(noisy))), 1.0); clean, noise, noisy = clean / peak, noise / peak, noisy / peak
            item = f'{i:04d}_snr{snr}'
            cp, npth, yp = f'clean/{item}.wav', f'noise/{item}.wav', f'noisy/{item}.wav'
            sf.write(OUT / cp, clean, rate, subtype='PCM_16'); sf.write(OUT / npth, noise, rate, subtype='PCM_16'); sf.write(OUT / yp, noisy, rate, subtype='PCM_16')
            writer.writerow({'id': item, 'clean': cp, 'noise': npth, 'noisy': yp, 'snr_db': snr, 'source_clean': row['path'], 'source_noise': str(npath.relative_to(MUSAN_NOISE))})
print('Готово:', manifest)

In [ ]:
# Финальная проверка: сколько получилось и один пример для прослушивания.
from IPython.display import Audio, display
print('Шумов:', sum(1 for _ in MUSAN_NOISE.rglob('*.wav')))
print('Пар:', sum(1 for _ in (DATA_ROOT / 'derived' / 'denoise_eval' / 'noisy').glob('*.wav')))
display(Audio(str(DATA_ROOT / 'derived' / 'denoise_eval' / 'noisy' / '0000_snr0.wav')))